SummarizationMiddleware 摘要中间件
HumanInTheLoopMiddleware：人工介入中间件
ToolCallLimitMiddleware：限制工具调用次数中间件
....

基类：AgentMiddleware：所有中间件都继承该类，如果要自定义中间件则继承这类


In [8]:
import datetime
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, before_model, after_model
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

llm = init_chat_model(
    model="glm-5.3-flash",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)


prompt  = """
    你是一个通用助手
"""

messages = [
    SystemMessage(content="你是一个专业的 LangChain 助手，回答简洁、准确、适合初学者。"),

    # 第 1 轮
    HumanMessage(content="你好，能简单介绍一下 LangChain 吗？"),
    AIMessage(content="LangChain 是一个用于构建大语言模型应用的框架，提供模型、提示、链、记忆、检索、代理等组件。"),

    # 第 2 轮
    HumanMessage(content="LangChain 里的消息对象有哪些类型？"),
    AIMessage(content="常见的有 SystemMessage、HumanMessage、AIMessage、ToolMessage、FunctionMessage、ChatMessage 等。"),

    # 第 3 轮
    HumanMessage(content="SystemMessage 和 HumanMessage 有什么区别？"),
    AIMessage(content="SystemMessage 用于设定系统指令或角色，HumanMessage 表示用户输入。"),

    # 第 4 轮
    HumanMessage(content="AIMessage 可以包含工具调用吗？"),
    AIMessage(content="可以。AIMessage 可以通过 tool_calls 字段表示模型请求调用工具。"),

    # 第 5 轮
    HumanMessage(content="怎么把消息列表传给模型？"),
    AIMessage(content="先构建 messages 列表，然后调用 model.invoke(messages)，例如 ChatOpenAI 的 invoke 方法。"),

    # 第 6 轮
    HumanMessage(content="如果要做多轮对话，需要保存历史吗？"),
    AIMessage(content="需要。把历史 HumanMessage 和 AIMessage 按顺序加入消息列表，或者使用 LangChain 的记忆组件。"),

    # 第 7 轮
    HumanMessage(content="LangChain 的 LCEL 是什么？"),
    AIMessage(content="LCEL 是 LangChain Expression Language，可以用管道符组合组件，例如 prompt | model | parser。"),

    # 第 8 轮
    HumanMessage(content="RAG 在 LangChain 中怎么实现？"),
    AIMessage(content="通常用文档加载器、文本分割器、向量存储、检索器和模型链组合实现 RAG。"),

    # 第 9 轮
    HumanMessage(content="Agent 和 Chain 有什么区别？"),
    AIMessage(content="Chain 通常是固定流程，Agent 则由模型动态决定调用哪些工具以及执行步骤。"),

    # 第 10 轮
    HumanMessage(content="最后，给我一个学习 LangChain 的建议吧。"),
    AIMessage(content="先掌握消息对象、提示模板和 LCEL，再做 RAG 和 Agent 小项目，边写边查官方文档。"),
]

summary_prompt = """你是对话摘要压缩器。请把下面的历史对话压缩成极简摘要，供另一个模型继续对话使用。

规则：
1. 只保留：用户目标、已确认事实、关键结论、重要决定、待办事项、用户偏好、专有名词、代码、API、参数、数字、错误信息。
2. 删除：寒暄、重复、客套、解释过程、无关细节。
3. 不编造、不推测、不添加原文没有的信息。
4. 默认用中文，不要标题、不要分节，直接一段话或最多 5 条短句。
5. 摘要必须明显短于原文；如果原文很短，可以只保留一句话。

历史对话：
{messages}
"""


graph = create_agent(
    model=llm,
    # system_prompt= prompt,
    # 给智能体起个名字
    name="通用智能体",
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",200),
            keep=("messages",2), # 保留2条
            summary_prompt=summary_prompt) # 自定义摘要提示词
    ]
)

response = graph.invoke({
    "messages": messages
})

rprint(response)






{
    'messages': [
        HumanMessage(
            content='Here is a summary of the conversation to date:\n\n用户是初学者，在学习 
LangChain，助手需简洁、准确、适合初学者。已确认要点：消息类型有 
SystemMessage、HumanMessage、AIMessage、ToolMessage、FunctionMessage、ChatMessage；SystemMessage 
设定系统指令/角色，HumanMessage 为用户输入；AIMessage 可通过 tool_calls 字段请求调用工具；传消息用 
model.invoke(messages)（如 ChatOpenAI）；多轮对话需按顺序保存历史消息或用记忆组件；LCEL（LangChain Expression 
Language）用管道符组合组件，如 prompt | model | parser；RAG 
由文档加载器、文本分割器、向量存储、检索器和模型链组合实现；Chain 是固定流程，Agent 
由模型动态决定调用哪些工具及步骤。',
            additional_kwargs={'lc_source': 'summarization'},
            response_metadata={},
            id='e8e31d2e-3a07-4892-bcc3-86cee676ae52'
        ),
        HumanMessage(
            content='最后，给我一个学习 LangChain 的建议吧。',
            additional_kwargs={},
            response_metadata={},
            id='49d0189b-79b2-4288-b632-c41bef83f23c'
        ),
        AIMessage(
            content='先掌握消息对象、提示模板和 LCEL，再做 RAG 和 Agent 小项目，边写边查官方文档。',
            additional_kwargs={},
            response_metadata={},
            id='75d8be02-41c1-4d09-a271-c0ff6d7f2561',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        AIMessage(
            content='没错，这个路线很合理！帮你把它整理成更具体的步骤：\n\n**第一阶段：打好基础**\n1. 
消息对象（System/Human/AI Message）和 `model.invoke()` 的用法\n2. 
提示模板（`ChatPromptTemplate`）和输出解析器（`StrOutputParser`）\n3. 用 LCEL 把它们串起来：`prompt | model | 
parser`\n\n**第二阶段：做 RAG 小项目**\n- 加载文档 → 分割文本 → 向量化存入向量库 → 检索 → 结合问题让模型回答\n- 
建议用一个小 PDF 或几篇文档练手，能跑通就有成就感了\n\n**第三阶段：尝试 Agent**\n- 
定义一两个简单工具（比如计算器、天气查询），观察模型的 `tool_calls` 是如何触发工具的\n\n**两个实用提醒：**\n- 
LangChain 版本迭代很快，网上旧教程的写法可能已过时，遇到报错优先查官方文档（python.langchain.com）\n- 多用 
`verbose=True` 或打印中间结果，看清每一步的数据流转，比死记概念有效得多\n\n祝你学习顺利，动手写代码是最好的老师！',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 948,
                    'prompt_tokens': 216,
                    'total_tokens': 1164,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 0,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': 0,
                        'text_tokens': 0
                    },
                    'usage_semantic': 'openai',
                    'usage_source': 'anthropic',
                    'billing_usage': {
                        'source': 'claude_messages',
                        'semantic': 'anthropic',
                        'claude_usage': {
                            'input_tokens': 216,
                            'cache_creation_input_tokens': 0,
                            'cache_read_input_tokens': 0,
                            'output_tokens': 948,
                            'claude_cache_creation_5_m_tokens': 0,
                            'claude_cache_creation_1_h_tokens': 0,
                            'server_tool_use': {'web_search_requests': 0}
                        }
                    },
                    'input_tokens': 216,
                    'output_tokens': 0,
                    'input_tokens_details': None,
                    'claude_cache_creation_5_m_tokens': 0,
                    'claude_cache_creation_1_h_tokens': 0
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3-flash',
                'system_fingerprint': None,
                'id': 'msg_20260929140339afb1800f50994a80',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='通用智能体',
          

案例：人工审核的邮件发送智能体


In [12]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from dotenv import load_dotenv


load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

# 初始化模型（以 OpenAI 为例，也可替换为其他兼容模型）
llm = init_chat_model(
    # model="deepseek-v4-pro",
    model="glm-5.3-flash",
    # glm-5.3-flash
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送邮件到指定地址。"""
    # 实际场景中这里会连接 SMTP 服务发送邮件
    return f"邮件已发送至 {to}，主题：{subject}"

@tool
def read_calendar(date: str) -> str:
    """读取指定日期的日历事件。"""
    return f"{date} 的日历事件：10:00 团队会议"


# agent = create_agent(
#         model=llm,
#         tools=[send_email, read_calendar]
#     )
#
# response = agent.invoke({
#     "messages": "你是谁？"
# })
#
# print(response)

# 必须使用 checkpointer 来保存中断状态，否则无法恢复执行[reference:2]
checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[send_email, read_calendar],
    system_prompt="你是一个邮件助手，可以帮助用户发送邮件和查看日历。",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # send_email 需要审核，允许所有决策类型
                "send_email": True,
                # read_calendar 安全，无需审核
                "read_calendar": False,
            }
        )
    ],
    checkpointer=checkpointer,  # 必填
)



# 配置线程 ID，用于恢复时定位同一个会话
config = {"configurable": {"thread_id": "email-thread-1"}}

# 用户请求发送邮件
result = agent.invoke(
    {"messages": [{"role": "user", "content": "帮我给 boss@company.com 发一封邮件，主题是'周报'，内容是'本周工作已完成。'"}]},

    config=config
)

# 检查是否触发了中断
if "__interrupt__" in result:
    interrupt_request = result["__interrupt__"][0]
    print("触发了人工审核中断：")
    print(interrupt_request.value)  # 包含待审核的工具调用信息
    # 再次调用
    result = agent.invoke(

    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config)
    rprint(result)

else:
    print("没有中断，直接执行完成：", result["messages"][-1].content)




触发了人工审核中断：
{'action_requests': [{'name': 'send_email', 'args': {'body': '本周工作已完成。', 'subject': '周报', 'to': 'boss@company.com'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': '本周工作已完成。', 'subject': '周报', 'to': 'boss@company.com'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}


{
    'messages': [
        HumanMessage(
            content="帮我给 boss@company.com 发一封邮件，主题是'周报'，内容是'本周工作已完成。'",
            additional_kwargs={},
            response_metadata={},
            id='dd0592f8-af61-438e-a41d-d67ff5106368'
        ),
        AIMessage(
            content='我来帮您发送这封邮件。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 81,
                    'prompt_tokens': 267,
                    'total_tokens': 348,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 0,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': 0,
                        'text_tokens': 0
                    },
                    'usage_semantic': 'openai',
                    'usage_source': 'anthropic',
                    'billing_usage': {
                        'source': 'claude_messages',
                        'semantic': 'anthropic',
                        'claude_usage': {
                            'input_tokens': 267,
                            'cache_creation_input_tokens': 0,
                            'cache_read_input_tokens': 0,
                            'output_tokens': 81,
                            'claude_cache_creation_5_m_tokens': 0,
                            'claude_cache_creation_1_h_tokens': 0,
                            'server_tool_use': {'web_search_requests': 0}
                        }
                    },
                    'input_tokens': 267,
                    'output_tokens': 0,
                    'input_tokens_details': None,
                    'claude_cache_creation_5_m_tokens': 0,
                    'claude_cache_creation_1_h_tokens': 0
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3-flash',
                'system_fingerprint': None,
                'id': 'msg_202609291738126bf20979367e4091',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0ec86-c5c0-7a43-98a3-bcdc88c3f700-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'body': '本周工作已完成。', 'subject': '周报', 'to': 'boss@company.com'},
                    'id': 'call_62d029d37d714f6297c0a5e6',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 267,
                'output_tokens': 81,
                'total_tokens': 348,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='邮件已发送至 boss@company.com，主题：周报',
            name='send_email',
            id='be5f2158-98ad-4466-87be-464f94f6bcfc',
            tool_call_id='call_62d029d37d714f6297c0a5e6'
        ),
        AIMessage(
            content='邮件已成功发送！📧\n\n**发送详情：**\n- **收件人：** boss@company.com\n- **主题：** 周报\n- 
**内容：** 本周工作已完成。\n\n还有其他需要帮忙的吗？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 64,
                    'prompt_tokens': 323,
                    'total_tokens': 387,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
  

In [5]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

# 初始化模型（以 OpenAI 为例，也可替换为其他兼容模型）
llm = init_chat_model(
    # model="deepseek-v4-pro",
    model="glm-5.3-flash",
    # glm-5.3-flash
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)


agent = create_agent(
    model=llm,
    # system_prompt="你是一个邮件助手，可以帮助用户发送邮件和查看日历。",
    middleware=[

        TodoListMiddleware()
    ],
)

config = {"configurable": {"thread_id": "trip-1"}}

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "帮我调研杭州国庆期间的天气、高铁余票、热门酒店价格，然后综合这些信息给我一份 7 天行程建议"}]},
    # config=config,
    stream_mode="updates",   # ← 关键：每步输出增量
):
    # chunk 是 dict: {节点名: {状态字段: 新值}}
    print("=== chunk ===")
    print(chunk)

    # 只看 todos 变更
    if "todos" in str(chunk):
        for node, updates in chunk.items():
            if isinstance(updates, dict) and "todos" in updates:
                print(f"[{node}] todos 更新：")
                for todo in updates["todos"]:
                    status_icon = {
                        "pending": "⏳",
                        "in_progress": "🔄",
                        "completed": "✅",
                    }.get(todo["status"], "❓")
                    print(f"  {status_icon} {todo['content']} — {todo['status']}")


# response = agent.invoke({
#     # 帮我调研杭州国庆期间的天气、高铁余票、热门酒店价格，然后综合这些信息给我一份 7 天行程建议
#     "messages": [{"role":"user","content":"帮我调研杭州国庆期间的天气、高铁余票、热门酒店价格，然后综合这些信息给我一份 7 天行程建议"}]
#     # "messages": [{"role":"user","content":"hi"}]
# })

rprint(response)



=== chunk ===
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 996, 'prompt_tokens': 1391, 'total_tokens': 2387, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': 0, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': 0, 'text_tokens': 0}, 'usage_semantic': 'openai', 'usage_source': 'anthropic', 'billing_usage': {'source': 'claude_messages', 'semantic': 'anthropic', 'claude_usage': {'input_tokens': 1391, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 996, 'claude_cache_creation_5_m_tokens': 0, 'claude_cache_creation_1_h_tokens': 0, 'server_tool_use': {'web_search_requests': 0}}}, 'input_tokens': 1391, 'output_tokens': 0, 'input_tokens_details': None, 'claude_cache_creation_5_m_toke

{
    'messages': [
        HumanMessage(
            content='帮我调研杭州国庆期间的天气、高铁余票、热门酒店价格，然后综合这些信息给我一份 7 天行程建议',
            additional_kwargs={},
            response_metadata={},
            id='1f397491-7aec-4eec-9e6b-9e9b2b9c4e90'
        ),
        AIMessage(
            content='先说明一点：我目前**无法联网查询实时数据**（实时天气预报、12306 
余票、酒店实时报价），所以下面的天气和价格部分基于杭州常年气候规律和国庆假期的普遍行情给出参考区间，并附上你自行核
实实时信息的渠道。行程建议则是完整可用的。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 1999,
                    'prompt_tokens': 1391,
                    'total_tokens': 3390,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 0,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': 0,
                        'text_tokens': 0
                    },
                    'usage_semantic': 'openai',
                    'usage_source': 'anthropic',
                    'billing_usage': {
                        'source': 'claude_messages',
                        'semantic': 'anthropic',
                        'claude_usage': {
                            'input_tokens': 1391,
                            'cache_creation_input_tokens': 0,
                            'cache_read_input_tokens': 0,
                            'output_tokens': 1999,
                            'claude_cache_creation_5_m_tokens': 0,
                            'claude_cache_creation_1_h_tokens': 0,
                            'server_tool_use': {'web_search_requests': 0}
                        }
                    },
                    'input_tokens': 1391,
                    'output_tokens': 0,
                    'input_tokens_details': None,
                    'claude_cache_creation_5_m_tokens': 0,
                    'claude_cache_creation_1_h_tokens': 0
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3-flash',
                'system_fingerprint': None,
                'id': 'msg_2026092916184299b2d413751747ee',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0ec3e-055f-7fd0-a4e1-622e6f882a71-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': '整理杭州国庆期间天气规律及穿衣/防雨建议', 'status': 'in_progress'},
                            {'content': '给出高铁购票策略与余票查询指引', 'status': 'pending'},
                            {'content': '整理国庆杭州热门酒店价格区间参考', 'status': 'pending'},
                            {'content': '输出 7 天行程建议（含错峰与预约提示）', 'status': 'pending'}
                        ]
                    },
                    'id': 'call_e38ab59553b44a26b22707c6',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1391,
                'output_tokens': 1999,
                'total_tokens': 3390,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': '整理杭州国庆期间天气规律及穿衣/防雨建议', 'status': 
'in_progress'}, {'content': '给出高铁购票策略与余票查询指引', 'status': 'pending'}, {'content': 
'整理国庆杭州热门酒店价格区间参考', 'status': 'pending'}, {'content': '输出 7 天行程建议（含错峰与预约提示）', 
'status': 'pending'}]"

In [19]:
from typing import Any
from langgraph.runtime import Runtime
import os
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import SystemMessage,AIMessage
from langchain.agents.middleware import before_model,after_model
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import datetime

current_time = datetime.datetime.now().isoformat()

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

# 初始化模型（以 OpenAI 为例，也可替换为其他兼容模型）
llm = init_chat_model(
    # model="deepseek-v4-pro",
    model="glm-5.3-flash",
    # glm-5.3-flash
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)



# 自定义中间件：
@before_model
def add_system_time(state:AgentState,runtime:Runtime) -> dict:
    """每次调用模型之前都会注入当前系统时间"""
    current_time = datetime.datetime.now().isoformat()
    print("当前时间：" + current_time)
    # 返回值会合并到Agent的状态中？这个Agent的状态是什么？
    return {
        "messages":[SystemMessage(content=f"当前系统时间：{current_time}")]
    }


@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """每次模型调用完成后打印日志。"""
    # 取最后一条消息，就是模型刚返回的 AIMessage
    last_msg = state["messages"][-1]

    # 安全判断：确保是 AIMessage
    if isinstance(last_msg, AIMessage):
        content_preview = (last_msg.content or "")[:80].replace("\n", " ")
        tool_calls = [tc["name"] for tc in (last_msg.tool_calls or [])]
        usage = last_msg.usage_metadata or {}

        print("=" * 60)
        print(f"[模型调用完成]")
        print(f"  内容预览: {content_preview!r}")
        if tool_calls:
            print(f"  工具调用: {tool_calls}")
        if usage:
            print(f"  Token用量: input={usage.get('input_tokens')}, "
                  f"output={usage.get('output_tokens')}, "
                  f"total={usage.get('total_tokens')}")
        print("=" * 60)

    # 返回 None 表示不修改状态
    return None



agent = create_agent(
    model=llm,
    # system_prompt="你是一个邮件助手，可以帮助用户发送邮件和查看日历。",
    middleware=[
        add_system_time,
        log_after_model
    ],
)


response = agent.invoke(
    {
        "messages": [{"role": "user", "content": "你好"}]
    }
)

rprint(response)




当前时间：2026-09-29T18:02:20.074203
[模型调用完成]
  内容预览: '你好！👋 很高兴见到你。我是 GLM，由 Z.ai 训练的大语言模型。  今天我能为你做些什么？无论是回答问题、帮助写作、分析信息，还是聊聊任何话题，我都很乐意'
  Token用量: input=34, output=142, total=176


{
    'messages': [
        HumanMessage(
            content='你好',
            additional_kwargs={},
            response_metadata={},
            id='e2fd8148-ab8c-463d-8aa5-6749d461674e'
        ),
        SystemMessage(
            content='当前系统时间：2026-09-29T18:02:20.074203',
            additional_kwargs={},
            response_metadata={},
            id='201dfbf2-9994-4a0c-a094-bf98e3e212f3'
        ),
        AIMessage(
            content='你好！👋 很高兴见到你。我是 GLM，由 Z.ai 
训练的大语言模型。\n\n今天我能为你做些什么？无论是回答问题、帮助写作、分析信息，还是聊聊任何话题，我都很乐意帮忙。'
,
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 142,
                    'prompt_tokens': 34,
                    'total_tokens': 176,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 0,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': 0,
                        'text_tokens': 0
                    },
                    'usage_semantic': 'openai',
                    'usage_source': 'anthropic',
                    'billing_usage': {
                        'source': 'claude_messages',
                        'semantic': 'anthropic',
                        'claude_usage': {
                            'input_tokens': 34,
                            'cache_creation_input_tokens': 0,
                            'cache_read_input_tokens': 0,
                            'output_tokens': 142,
                            'claude_cache_creation_5_m_tokens': 0,
                            'claude_cache_creation_1_h_tokens': 0,
                            'server_tool_use': {'web_search_requests': 0}
                        }
                    },
                    'input_tokens': 34,
                    'output_tokens': 0,
                    'input_tokens_details': None,
                    'claude_cache_creation_5_m_tokens': 0,
                    'claude_cache_creation_1_h_tokens': 0
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3-flash',
                'system_fingerprint': None,
                'id': 'msg_2026092918030072f982c933bc4c89',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a0ec9d-802e-7471-a513-9ed13c4a4a38-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 34,
                'output_tokens': 142,
                'total_tokens': 176,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}